# Lakehouse Shortcut Audit: Multipart Delta Checkpoints

This notebook checks whether the **current Fabric workspace** has lakehouse table shortcuts that point to Delta tables with multipart checkpoints (`_last_checkpoint.parts > 1`).

If multipart usage is found, the notebook prints the matching shortcuts.

In [10]:
import json
from concurrent.futures import ThreadPoolExecutor, as_completed

from sempy.fabric import FabricRestClient
import requests

try:
    import notebookutils
    runtime_context = notebookutils.runtime.context
    fs = notebookutils.fs

    def get_current_workspace_id():
        return runtime_context.get("currentWorkspaceId")

    def get_fabric_token():
        return notebookutils.credentials.getToken("https://api.fabric.microsoft.com")
except Exception:
    import mssparkutils
    runtime_context = mssparkutils.runtime.context()
    fs = mssparkutils.fs

    def get_current_workspace_id():
        try:
            return runtime_context["currentWorkspaceId"]
        except Exception:
            return None

    def get_fabric_token():
        return mssparkutils.credentials.getToken("https://api.fabric.microsoft.com")

StatementMeta(, ad5e9297-5c65-4e04-941f-aef866177167, 17, Finished, Available, Finished)

In [ ]:
client = None
WORKSPACE_ID = get_current_workspace_id()
MAX_WORKERS = 16
DEBUG = 1
HTTP_TIMEOUT_SECONDS = 15

if not WORKSPACE_ID:
    raise ValueError("Workspace ID not found in runtime context. Run inside Fabric workspace notebook.")

def get_client():
    global client
    if client is None:
        client = FabricRestClient()
    return client

def get_paged(path: str):
    return list(get_client().get_paged(path))

def list_lakehouses(workspace_id: str):
    items = get_paged(f"/v1/workspaces/{workspace_id}/items")
    return [item for item in items if str(item.get("type", "")).lower().startswith("lakehouse")]

def _auth_header_from_token(token: str):
    if token and token.startswith("Bearer "):
        return {"Authorization": token}
    return {"Authorization": f"Bearer {token}"}

def list_shortcuts_rest(workspace_id: str, item_id: str):
    token = get_fabric_token()
    headers = _auth_header_from_token(token)
    base_url = f"https://api.fabric.microsoft.com/v1/workspaces/{workspace_id}/items/{item_id}/shortcuts"

    results = []
    continuation_token = None

    while True:
        params = {}
        if continuation_token:
            params["continuationToken"] = continuation_token

        response = requests.get(base_url, headers=headers, params=params, timeout=HTTP_TIMEOUT_SECONDS)
        if response.status_code >= 400:
            response.raise_for_status()

        payload = response.json() if response.text else {}
        page_items = payload.get("value") or []
        results.extend(page_items)

        continuation_token = payload.get("continuationToken")
        if not continuation_token:
            break

    return results

def list_shortcuts(workspace_id: str, item_id: str):
    rest_error = None
    try:
        return list_shortcuts_rest(workspace_id, item_id)
    except Exception as ex:
        rest_error = ex

    try:
        return get_paged(f"/v1/workspaces/{workspace_id}/items/{item_id}/shortcuts")
    except Exception:
        if rest_error is not None:
            raise rest_error
        raise

def get_shortcut_full_path(shortcut: dict):
    shortcut_path = str(shortcut.get("path") or "").strip().replace("\\", "/")
    shortcut_name = str(shortcut.get("name") or "").strip().replace("\\", "/")

    if shortcut_path and shortcut_name:
        base = shortcut_path.rstrip("/")
        if base.lower().endswith(f"/{shortcut_name.lower()}") or base.lower() == shortcut_name.lower():
            combined = base
        else:
            combined = f"{base}/{shortcut_name}"
    elif shortcut_path:
        combined = shortcut_path
    else:
        combined = shortcut_name

    combined = combined.strip().lstrip("/")
    return combined if combined else None

def checkpoint_uri(workspace_id: str, lakehouse_item_id: str, table_rel_path: str):
    return (
        f"abfss://{workspace_id}@onelake.dfs.fabric.microsoft.com/"
        f"{lakehouse_item_id}/{table_rel_path}/_delta_log/_last_checkpoint"
    )

def read_checkpoint_parts_for_shortcut(workspace_id: str, lakehouse_item_id: str, table_rel_path: str):
    uri = checkpoint_uri(workspace_id, lakehouse_item_id, table_rel_path)
    try:
        raw = fs.head(uri, 2_000_000)
        checkpoint = json.loads(raw)
        return int(checkpoint.get("parts", 0)), uri
    except Exception:
        return None, uri

StatementMeta(, ad5e9297-5c65-4e04-941f-aef866177167, 14, Finished, Available, Finished)

StatementMeta(, ad5e9297-5c65-4e04-941f-aef866177167, 18, Finished, Available, Finished)

In [12]:
shortcut_refs = []  # (lakehouse_name, lakehouse_id, shortcut_path, key)
targets = set()     # (lakehouse_id, shortcut_path)

diagnostics = {
    "lakehouses_seen": 0,
    "lakehouses_shortcuts_read_ok": 0,
    "lakehouses_shortcuts_read_failed": 0,
    "shortcut_list_errors": 0,
    "shortcuts_returned": 0,
    "filtered_non_table_shortcuts": 0,
    "shortcut_list_error_details": [],
    "path_debug": [],
    "checkpoint_debug": [],
}

for lakehouse in list_lakehouses(WORKSPACE_ID):
    diagnostics["lakehouses_seen"] += 1
    lakehouse_id = lakehouse.get("id")
    lakehouse_name = lakehouse.get("displayName", "")
    if not lakehouse_id:
        continue

    try:
        shortcuts = list_shortcuts(WORKSPACE_ID, lakehouse_id)
        diagnostics["lakehouses_shortcuts_read_ok"] += 1
    except Exception as ex:
        diagnostics["shortcut_list_errors"] += 1
        diagnostics["lakehouses_shortcuts_read_failed"] += 1
        diagnostics["shortcut_list_error_details"].append(
            f"{lakehouse_name or lakehouse_id}: {type(ex).__name__}: {ex}"
        )
        continue

    diagnostics["shortcuts_returned"] += len(shortcuts)

    for shortcut in shortcuts:
        table_path = get_shortcut_full_path(shortcut)
        raw_path = str(shortcut.get("path") or "")
        shortcut_name = str(shortcut.get("name") or "")
        target_type = str((shortcut.get("target") or {}).get("type") or "")

        if not table_path:
            diagnostics["filtered_non_table_shortcuts"] += 1
            if DEBUG:
                diagnostics["path_debug"].append(
                    f"FILTER shortcut_without_path | lakehouse={lakehouse_name or lakehouse_id} | path={raw_path} | name={shortcut_name} | targetType={target_type}"
                )
            continue

        key = (lakehouse_id, table_path)
        targets.add(key)
        shortcut_refs.append((lakehouse_name, lakehouse_id, table_path, key))

        if DEBUG:
            diagnostics["path_debug"].append(
                f"CHECK shortcut | lakehouse={lakehouse_name or lakehouse_id} | path={raw_path} | name={shortcut_name} | fullPath={table_path} | targetType={target_type}"
            )

target_parts = {}

def check_one(target_key):
    lakehouse_item_id, table_path = target_key
    parts, uri = read_checkpoint_parts_for_shortcut(WORKSPACE_ID, lakehouse_item_id, table_path)
    return target_key, parts, uri

if targets:
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = [executor.submit(check_one, key) for key in targets]
        for future in as_completed(futures):
            key, parts, uri = future.result()
            target_parts[key] = parts
            if DEBUG:
                diagnostics["checkpoint_debug"].append(
                    f"CHECKPOINT {uri} | parts={parts if parts is not None else 'not_found'}"
                )

StatementMeta(, d43c7844-5cb3-4a02-a99c-7494821057e5, 5, Finished, Available, Finished)

StatementMeta(, ad5e9297-5c65-4e04-941f-aef866177167, 10, Finished, Available, Finished)

StatementMeta(, ad5e9297-5c65-4e04-941f-aef866177167, 15, Finished, Available, Finished)

StatementMeta(, ad5e9297-5c65-4e04-941f-aef866177167, 19, Finished, Available, Finished)

In [ ]:
multipart_shortcuts = []

for lakehouse_name, lakehouse_id, shortcut_path, key in shortcut_refs:
    parts = target_parts.get(key)
    if parts is not None and parts > 1:
        multipart_shortcuts.append((lakehouse_name, shortcut_path))

print(f"Workspace: {WORKSPACE_ID}")
print(f"Lakehouses scanned: {diagnostics.get('lakehouses_seen', 0)}")
print(f"Lakehouses with shortcut read success: {diagnostics.get('lakehouses_shortcuts_read_ok', 0)}")
print(f"Lakehouses with shortcut read errors: {diagnostics.get('lakehouses_shortcuts_read_failed', 0)}")
print(f"Shortcuts returned by API: {diagnostics.get('shortcuts_returned', 0)}")
print(f"Filtered (shortcuts without usable path): {diagnostics.get('filtered_non_table_shortcuts', 0)}")
print(f"Scanned shortcuts: {len(shortcut_refs)}")
print(f"Unique shortcut paths checked: {len(targets)}")
print(f"Shortcuts pointing to multipart checkpoints (parts > 1): {len(multipart_shortcuts)}")

had_shortcut_scope_errors = diagnostics.get("shortcut_list_errors", 0) > 0

if len(shortcut_refs) == 0:
    error_details = diagnostics.get("shortcut_list_error_details") or []
    if error_details:
        print("\nShortcut list error details:")
        for message in error_details:
            print(f"- {message}")

if DEBUG:
    print("\nPath debug:")
    for line in diagnostics.get("path_debug") or []:
        print(f"- {line}")

    print("\nCheckpoint debug:")
    for line in diagnostics.get("checkpoint_debug") or []:
        print(f"- {line}")

if multipart_shortcuts:
    print("\nMultipart checkpoint usage detected (Lakehouse | ShortcutPath):")
    for lakehouse_name, shortcut_path in multipart_shortcuts:
        print(f"{lakehouse_name} | {shortcut_path}")
elif had_shortcut_scope_errors:
    print("\nAudit incomplete: shortcut metadata could not be read for one or more lakehouses due to permission scope errors.")
    print("Grant required Fabric API scopes/permissions for reading lakehouse shortcuts, then rerun.")
else:
    print("No multipart checkpoint usage detected in scanned shortcuts.")

StatementMeta(, ad5e9297-5c65-4e04-941f-aef866177167, 20, Finished, Available, Finished)

Workspace: ec8f7acc-85d3-4805-a75b-631889070302
Lakehouses scanned: 3
Lakehouses with shortcut read success: 3
Lakehouses with shortcut read errors: 0
Shortcuts returned by API: 6
Filtered (shortcuts without usable path): 0
Scanned shortcuts: 6
Unique shortcut paths checked: 6
Shortcuts pointing to multipart checkpoints (parts > 1): 0

Path debug:
- CHECK shortcut | lakehouse=gitlh1 | path=/Tables | name=yellow_small | fullPath=Tables/yellow_small | targetType=AdlsGen2
- CHECK shortcut | lakehouse=gitlh1 | path=/Tables | name=loadtest_target | fullPath=Tables/loadtest_target | targetType=OneLake
- CHECK shortcut | lakehouse=gitlh1 | path=/Tables | name=delta1 | fullPath=Tables/delta1 | targetType=AdlsGen2
- CHECK shortcut | lakehouse=gitlh2 | path=/Tables/dbo | name=loadtest_target | fullPath=Tables/dbo/loadtest_target | targetType=OneLake
- CHECK shortcut | lakehouse=gitlh2 | path=/Tables/dbo | name=delta1 | fullPath=Tables/dbo/delta1 | targetType=AdlsGen2
- CHECK shortcut | lakehouse

In [14]:
checkpoint_path = "abfss://ec8f7acc-85d3-4805-a75b-631889070302@onelake.dfs.fabric.microsoft.com/3bdc6b5b-4ac0-4bdf-befd-57016bf36883/Tables/dbo/Delta_Multipart2/_delta_log/_last_checkpoint"
print(fs.head(checkpoint_path, 2_000_000))

StatementMeta(, ad5e9297-5c65-4e04-941f-aef866177167, 21, Finished, Available, Finished)

{"version":10,"size":4858,"parts":4856,"sizeInBytes":59573538,"numOfAddFiles":4856,"checkpointSchema":{"type":"struct","fields":[{"name":"txn","type":{"type":"struct","fields":[{"name":"appId","type":"string","nullable":true,"metadata":{}},{"name":"version","type":"long","nullable":true,"metadata":{}},{"name":"lastUpdated","type":"long","nullable":true,"metadata":{}}]},"nullable":true,"metadata":{}},{"name":"add","type":{"type":"struct","fields":[{"name":"path","type":"string","nullable":true,"metadata":{}},{"name":"partitionValues","type":{"type":"map","keyType":"string","valueType":"string","valueContainsNull":true},"nullable":true,"metadata":{}},{"name":"size","type":"long","nullable":true,"metadata":{}},{"name":"modificationTime","type":"long","nullable":true,"metadata":{}},{"name":"dataChange","type":"boolean","nullable":true,"metadata":{}},{"name":"tags","type":{"type":"map","keyType":"string","valueType":"string","valueContainsNull":true},"nullable":true,"metadata":{}},{"name":"